<a href="https://colab.research.google.com/github/Andresg324/so101-vla-data-study/blob/main/so101_Data_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
!pip install "lerobot[smolvla]"
!pip install "lerobot[smolvla,dataset]"

In [ ]:
!pip install av

In [ ]:
from huggingface_hub import login
login()

import wandb
wandb.login()

In [1]:
CONDITION = "clean" # This will be changed per condition, ro randomized / recovery / color, and re-run for each
DATASET   = f"Andresg324/cube-pickup-{CONDITION}_<timestamp>" # Need to include the real time stamp once training has happend
OUTDIR    = f"outputs/train/smolvla_{CONDITION}"
MODELREPO = f"Andresg324/smolvla-cube-{CONDITION}"

In [ ]:
from huggingface_hub import HfApi
for d in HfApi().list_datasets(author="Andresg324"):
    print(d.id)

In [ ]:
!rm -rf {OUTDIR}

In [ ]:
!lerobot-train \
  --policy.path=lerobot/smolvla_base \
  --policy.push_to_hub=false \
  --dataset.repo_id={DATASET} \
  --rename_map='{"observation.images.overhead": "observation.images.camera1", "observation.images.wrist": "observation.images.camera2"}' \
  --batch_size=32 \
  --steps=10000 \
  --save_freq=2000 \
  --output_dir={OUTDIR} \
  --job_name=smolvla_{CONDITION} \
  --policy.device=cuda \
  --wandb.enable=true

In [ ]:
#See where the checkpoint has landed
!ls {OUTDIR}/checkpoints/

from huggingface_hub import HfApi
api = HfApi()
api.create_repo(f"{MODELREPO}", repo_type="model", exist_ok=True)
api.upload_folder(
    folder_path=f"{OUTDIR}/checkpoints/last/pretrained_model",
    repo_id=f"{MODELREPO}",
    repo_type="model",
)